In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib
try: get_ipython().run_line_magic("matplotlib","inline")
except Exception: matplotlib.use("Agg")
import matplotlib.pyplot as plt
try: from IPython.display import display, Image
except Exception:
    def display(o): print(o)
    Image=None

try: _anchor=Path(__file__).resolve()
except NameError: _anchor=Path.cwd()
for _p in (_anchor,*_anchor.parents):
    if (_p/"project_paths.py").exists(): sys.path.insert(0,str(_p)); break
from project_paths import project_root

ROOT=project_root()
CRUISES=["P45_06","P45_07"]
BIN_DB=2.0
MIN_PER_BIN=3
STEEP=0.5          # uM/m: a bin counts as "stratified" above this |gradient|
OUT_DIR=ROOT/"cruises"/CRUISES[-1]/"_audit"/"suna"; OUT_DIR.mkdir(parents=True,exist_ok=True)

def load_L2(cid):
    f=ROOT/"cruises"/cid/"L2"/"SUNA"/f"{cid}__suna_L2_colocated.csv"
    if not f.exists():
        print(f"  [{cid}] no L2 at {f}\n        run step00 -> step01 -> step02 for {cid} first (just change CRUISE_ID)")
        return None
    df=pd.read_csv(f, parse_dates=["time_utc"])
    for c in ("L1_clean","in_cast"):
        if df[c].dtype!=bool:
            df[c]=df[c].astype(str).str.strip().str.lower().isin(["true","1","1.0"])
    return df

L2={cid:load_L2(cid) for cid in CRUISES}
L2={k:v for k,v in L2.items() if v is not None}
for cid,df in L2.items():
    n=df[(df.in_cast)&(df.L1_clean)].cast.nunique()
    print(f"{cid}: {len(df)} frames, {n} casts with clean in-cast data")

In [ ]:
def cast_bins(df_cast, step=BIN_DB):
    d=df_cast[(df_cast.L1_clean)&(df_cast.pressure_db.notna())].copy()
    if d.empty: return pd.DataFrame()
    d["bin"]=(d.pressure_db//step).astype(int)
    g=d.groupby("bin")
    out=pd.DataFrame({
        "p":g.pressure_db.median(),
        "down":d[d.leg=="down"].groupby("bin").nitrate_uM.median(),
        "up":d[d.leg=="up"].groupby("bin").nitrate_uM.median(),
        "comb":g.nitrate_uM.median(),
        "n":g.size(),
    }).sort_values("p")
    return out

def cruise_pairs(df):
    """Across all casts: (|vertical gradient|, |down-up mismatch|) per usable bin."""
    rows=[]
    for cid,gc in df[(df.in_cast)&(df.L1_clean)].groupby("cast"):
        b=cast_bins(gc)
        if len(b)<3: continue
        p=b.p.to_numpy(); comb=b.comb.to_numpy()
        grad=np.full(len(b),np.nan)
        grad[1:-1]=(comb[2:]-comb[:-2])/(p[2:]-p[:-2])     # central diff, uM per db(~m)
        b=b.assign(grad=np.abs(grad))
        ok=b.down.notna()&b.up.notna()&b.grad.notna()&(b.n>=MIN_PER_BIN)
        for _,r in b[ok].iterrows():
            rows.append(dict(cast=cid,p=r.p,grad=r.grad,mism=abs(r.down-r.up)))
    return pd.DataFrame(rows)

def slope_through_origin(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y); x,y=x[m],y[m]
    if len(x)<3 or np.sum(x**2)==0: return np.nan,np.nan
    s=np.sum(x*y)/np.sum(x**2)
    ss_res=np.sum((y-s*x)**2); ss_tot=np.sum((y-y.mean())**2)
    return s,(1-ss_res/ss_tot if ss_tot>0 else np.nan)

In [ ]:
summary=[]; PAIRS={}
for cid,df in L2.items():
    P=cruise_pairs(df); PAIRS[cid]=P
    steep=P[P.grad>=STEEP]; s,r2=slope_through_origin(P.grad,P.mism)
    summary.append(dict(cruise=cid,
        casts=df[(df.in_cast)&(df.L1_clean)].cast.nunique(), bins=len(P),
        grad_med=round(P.grad.median(),2), grad_p90=round(P.grad.quantile(.9),2),
        grad_max=round(P.grad.max(),2), steep_bins=len(steep),
        mism_steep_med=round(steep.mism.median(),2) if len(steep) else np.nan,
        lag_slope=round(s,2), slope_R2=round(r2,2)))
S=pd.DataFrame(summary)
with pd.option_context("display.width",200): print(S.to_string(index=False))
print("\n  grad_*         how stratified the cruise was (uM/m); low grad_max = gentle gradients")
print("  mism_steep_med down/up disagreement in stratified bins = the lag, in uM")
print("  lag_slope      mismatch per unit gradient = the sensor's lag signature")

if {"P45_06","P45_07"}.issubset(set(S.cruise)):
    s06=float(S[S.cruise=="P45_06"].lag_slope.iloc[0]); s07=float(S[S.cruise=="P45_07"].lag_slope.iloc[0])
    denom=max(abs(s06),abs(s07)); rel=abs(s06-s07)/denom*100 if denom>0 else float("nan")
    print(f"\nVERDICT: lag signatures {s06} (P45_06) vs {s07} (P45_07) differ by {rel:.0f}%.")
    if rel < 25:
        print("  Same instrument behaviour on both cruises: the SUNA always flushed poorly.")
        print("  P45-06 looked fine only because its gradients were gentler and its calibration used one cast.")
    else:
        print("  Flushing differed between cruises: investigate fouling, mounting, or the pump.")
S.to_csv(OUT_DIR/"cruise_hysteresis_comparison.csv", index=False, encoding="utf-8-sig")

In [ ]:
colors={"P45_06":"#2E6FB0","P45_07":"#d62728"}
fig,ax=plt.subplots(1,2,figsize=(12,4.6))
for cid,P in PAIRS.items():
    c=colors.get(cid,"#888")
    ax[0].plot(P.grad,P.mism,".",ms=3,alpha=.35,color=c)
    s,r2=slope_through_origin(P.grad,P.mism)
    xx=np.linspace(0,P.grad.quantile(.98),50)
    ax[0].plot(xx,s*xx,"-",color=c,lw=2,label=f"{cid}: slope {s:.1f} (R2 {r2:.2f})")
ax[0].set_xlabel("|vertical gradient| (uM/m)"); ax[0].set_ylabel("|down - up| (uM)")
ax[0].set_title("Lag signature: hysteresis vs stratification"); ax[0].legend(fontsize=8,frameon=False)
for cid,P in PAIRS.items():
    ax[1].hist(P.grad,bins=np.arange(0,3.1,0.15),histtype="step",lw=2,color=colors.get(cid,"#888"),label=cid)
ax[1].axvline(STEEP,color="#999",ls="--",lw=1)
ax[1].set_xlabel("|vertical gradient| (uM/m)"); ax[1].set_ylabel("bins")
ax[1].set_title("How stratified was each cruise?"); ax[1].legend(fontsize=8,frameon=False)
for a in ax:
    for sp in ("top","right"): a.spines[sp].set_visible(False)
fig.tight_layout(); out=OUT_DIR/"cruise_hysteresis_comparison.png"; fig.savefig(out,dpi=135)
try: plt.close(fig)
except Exception: pass
if Image is not None:
    try: display(Image(filename=str(out)))
    except Exception: pass
print("saved",out.name)